## Get filenames for selected events

Use ens member and year to generate filename for downloading CESM2-LENS data for selected variables

In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import subprocess
import shutil
import glob

In [2]:
#selected extreme events
events_label = 'WestNO_events_04_05_26'

#list of events to extract
#events = ['precip_5_r25_2073-03-12_62.67_8.75.png', 'precip_5_r17_2051-06-01_60.79_9.38.png','precip_5_r22_2097-12-06_66.44_13.12.png']
#2050
#events = ['precip_5_r31_2053-02-01_66.44_12.5.png','precip_5_r6_2046-01-26_67.38_13.75.png','precip_5_r9_2046-12-28_67.38_13.75.png']

#selected for ens members compatible with WRF
events_WRF = ['PRECT5day_6_r35_2037-07-20_65.13_13.47_single.png',
'PRECT5day_6_r15_2083-06-25_63.61_12.5_single.png',
'PRECT5day_6_r17_2095-08-29_70.21_21.88_single.png',
'PRECT5day_6_r16_2040-08-17_69.74_20.42_single.png',
'PRECT5day_6_r14_2094-08-16_60.79_10.0_single.png',
'PRECT5day_6_r41_2052-03-12_59.84_10.0_single.png']

#******change event list here*******
events = ['PRECT5day_5_r24_2076-07-18_62.67_6.25_single.png',
'PRECT5day_5_r30_2020-05-12_61.73_6.25_single.png',
'PRECT5day_5_r49_2049-12-07_61.73_6.25_single.png',
'PRECT5day_5_r36_2096-06-12_62.67_6.25_single.png',
'PRECT5day_5_r31_2093-11-19_62.04_6.25_single.png']

In [3]:
event_table = pd.DataFrame(events,columns=['img_filename'])

In [4]:
#extract columns from filename
event_table['date'] = event_table.img_filename.str.split('_').str[3]
event_table['ens_member'] = event_table.img_filename.str.split('_').str[2]
event_table['member_ind'] = pd.to_numeric(event_table['ens_member'].str[1:]) - 1
event_table['lat'] = event_table.img_filename.str.split('_').str[4]
event_table['lon'] = event_table.img_filename.str[:-4].str.split('_').str[5]
event_table['year'] = pd.to_numeric(event_table.img_filename.str.split('_').str[3].str[:4])

In [ ]:
#import ens member list for indexing and making member_lab column
filelist = sorted(glob.glob(f'/div/no-backup/Large_Ensemble_data/CESM2-LENS/BSSP370smbb/PRECT/*h1*.nc'))

#experiment labels from filelist
ens_labs = [file.split('/')[-1][30:-34] for file in filelist]

ens_ind = np.arange(len(ens_labs))

ens_lab_dict = dict(zip(ens_ind, ens_labs))


#ens_lab_dict_file = open('/div/nac/users/zofias/XXN/ens_member_labs_dict.txt', 'a')
#ens_lab_dict_file.write(str(ens_lab_dict))
#ens_lab_dict_file.close()

In [6]:
event_table['member_lab'] = event_table['member_ind'].map(ens_lab_dict)

In [7]:
event_table

,img_filename,date,ens_member,member_ind,lat,lon,year,member_lab
0,PRECT5day_5_r24_2076-07-18_62.67_6.25_single.png,2076-07-18,r24,23,62.67,6.25,2076,1251.014
1,PRECT5day_5_r30_2020-05-12_61.73_6.25_single.png,2020-05-12,r30,29,61.73,6.25,2020,1251.020
2,PRECT5day_5_r49_2049-12-07_61.73_6.25_single.png,2049-12-07,r49,48,61.73,6.25,2049,1301.019
3,PRECT5day_5_r36_2096-06-12_62.67_6.25_single.png,2096-06-12,r36,35,62.67,6.25,2096,1281.016
4,PRECT5day_5_r31_2093-11-19_62.04_6.25_single.png,2093-11-19,r31,30,62.04,6.25,2093,1281.011


## Generate filenames for selected variables

In [8]:
# ****** Select vars here ********
vars = ['PRECT','U850','V850','TS','PSL']

# U850	time: mean	Zonal wind at 850 hPa	m/s	time lev lat lon
# V850	time: mean	Meridional wind at 850 hPa	m/s	time lev lat lon
# TS	time: mean	Surface temperature	K	time lev lat lon
# PSL	time: mean	Sea level pressure	Pa	time lev lat lon

In [9]:
#decade group categories (for SSP)
year_labs = ['20150101-20241231.nc','20250101-20341231.nc','20350101-20441231.nc','20450101-20541231.nc','20550101-20641231.nc','20650101-20741231.nc','20750101-20841231.nc','20850101-20941231.nc','20950101-21001231.nc']

In [10]:
bins = [2015, 2024, 2034, 2044, 2054, 2064, 2074, 2084, 2094,2101]
labels = year_labs
event_table['decade_lab'] = pd.cut(event_table.year, bins, labels = labels,include_lowest = True)

#print(event_table)

In [11]:
event_table

,img_filename,date,ens_member,member_ind,lat,lon,year,member_lab,decade_lab
0,PRECT5day_5_r24_2076-07-18_62.67_6.25_single.png,2076-07-18,r24,23,62.67,6.25,2076,1251.014,20750101-20841231.nc
1,PRECT5day_5_r30_2020-05-12_61.73_6.25_single.png,2020-05-12,r30,29,61.73,6.25,2020,1251.020,20150101-20241231.nc
2,PRECT5day_5_r49_2049-12-07_61.73_6.25_single.png,2049-12-07,r49,48,61.73,6.25,2049,1301.019,20450101-20541231.nc
3,PRECT5day_5_r36_2096-06-12_62.67_6.25_single.png,2096-06-12,r36,35,62.67,6.25,2096,1281.016,20950101-21001231.nc
4,PRECT5day_5_r31_2093-11-19_62.04_6.25_single.png,2093-11-19,r31,30,62.04,6.25,2093,1281.011,20850101-20941231.nc


In [20]:
# save locs file
event_table.to_csv(f'/div/nac/users/zofias/XXN/extreme_loc_tables/{events_label}_table.csv',index=True)

In [ ]:
#make filelist with all vars including address for downloading from server
filelist = []
loc = 'https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/'
for n,var in enumerate(vars):
    for index, row in event_table.iterrows():
        filename = f"{loc}{var}/b.e21.BSSP370smbb.f09_g17.LE2-{row['member_lab']}.cam.h1.{var}.{row['decade_lab']}"
        filelist.append(filename)

In [22]:
filelist

['https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/PRECT/b.e21.BSSP370smbb.f09_g17.LE2-1251.014.cam.h1.PRECT.20750101-20841231.nc',
 'https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/PRECT/b.e21.BSSP370smbb.f09_g17.LE2-1251.020.cam.h1.PRECT.20150101-20241231.nc',
 'https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/PRECT/b.e21.BSSP370smbb.f09_g17.LE2-1301.019.cam.h1.PRECT.20450101-20541231.nc',
 'https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/PRECT/b.e21.BSSP370smbb.f09_g17.LE2-1281.016.cam.h1.PRECT.20950101-21001231.nc',
 'https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/PRECT/b.e21.BSSP370smbb.f09_g17.LE2-1281.011.cam.h1.PRECT.20850101-20941231.nc',
 'https://osdf-data.gdex.ucar.edu/ncar/gdex/d651056/CESM2-LE/atm/proc/tseries/day_1/U850/b.e21.BSSP370smbb.f09_g17.LE2-1251.014.cam.h1.U850.20750101-20841231.nc',
 'https://os

In [23]:
#save file list
np.save('/div/no-backup-nac/users/zofias/CESM2-LENS/weather_data/filelist_to_read_cesm2lens.npy',filelist)